# Compare Geothermal Supply With Heat Demand

## 1. Imports


In [2]:
from pathlib import Path
import geopandas as gpd
import numpy as np
import pandas as pd
import gc
import math
from shapely.geometry import box

## 2. Project and file paths


In [3]:
PROJECT_DIR = next(candidate 
                   for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents] 
                   if (candidate / "01_Data").exists())

In [4]:
MODEL_OUTPUT_DIR = (PROJECT_DIR / "01_Data" / "Processed" / "Geothermal" / "model_outputs")

ENERGY_PROCESSED_DIR = (PROJECT_DIR / "01_Data" / "Processed" / "Energy")

# Closed-loop geothermal supply outputs
CLOSED_LOOP_GPKG = MODEL_OUTPUT_DIR / "02_closed_loop_supply.gpkg"
CLOSED_LOOP_LAYER = "closed_loop_supply"
BHE_GRID_LAYER = "bhe_grid_75m"

# Residential heat-demand output
HEAT_DEMAND_GPKG = (ENERGY_PROCESSED_DIR / "plymouth_heat_demand_2024.gpkg")

HEAT_DEMAND_LAYER = "lsoa_heat_demand_2024"

# Supply-demand comparison outputs
OUTPUT_GPKG = MODEL_OUTPUT_DIR / "06_supply_demand_comparison.gpkg"

LSOA_COMPARISON_CSV = (MODEL_OUTPUT_DIR / "06_lsoa_supply_demand_comparison.csv")

CITY_SUMMARY_CSV = (MODEL_OUTPUT_DIR / "06_supply_demand_city_summary.csv")

MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Supply file exists:", CLOSED_LOOP_GPKG.exists())
print("Demand file exists:", HEAT_DEMAND_GPKG.exists())

Supply file exists: True
Demand file exists: True


## 3. Load completed supply and demand outputs


In [5]:
closed_loop = gpd.read_file(CLOSED_LOOP_GPKG,layer=CLOSED_LOOP_LAYER)

bhe_grid = gpd.read_file(CLOSED_LOOP_GPKG,layer=BHE_GRID_LAYER)

heat_demand = gpd.read_file(HEAT_DEMAND_GPKG,layer=HEAT_DEMAND_LAYER)



print("Closed-loop geology polygons:",len(closed_loop))
print("Uniform-grid BHE points:",len(bhe_grid))
print("Heat-demand LSOAs:", len(heat_demand))

Closed-loop geology polygons: 328
Uniform-grid BHE points: 14056
Heat-demand LSOAs: 164


## 4. Check required input fields


In [6]:
# Heat-demand fields
heat_required_fields = ["LSOA_code","LSOA","total_useful_heat_mwh"]

# Geology polygon fields
geology_required_fields = ["geology_polygon_id","model_area_m2"]

# Uniform BHE grid fields
bhe_required_fields = ["bhe_id","grid_id","grid_spacing_m","model_unit_lithology_id",
                       "low_useful_heat_mwh_year","representative_useful_heat_mwh_year",
                       "high_useful_heat_mwh_year"
                       ]

In [7]:
# Check for missing fields
missing_heat_fields = [field for field in heat_required_fields if field not in heat_demand.columns]

missing_geology_fields = [field for field in geology_required_fields if field not in closed_loop.columns]

missing_bhe_fields = [field for field in bhe_required_fields if field not in bhe_grid.columns]


if missing_heat_fields:
    raise ValueError(f"Missing heat demand fields: {missing_heat_fields}")

if missing_geology_fields:
    raise ValueError(f"Missing geology fields: {missing_geology_fields}")

if missing_bhe_fields:
    raise ValueError(f"Missing BHE-grid fields: {missing_bhe_fields}")


## 6. Prepare geothermal geology for LSOA model area calculation

In [8]:
geology_for_area = closed_loop[["geology_polygon_id","model_area_m2","geometry"]].copy()

# Rename the model area column to avoid confusion with other area columns
geology_for_area = geology_for_area.rename(columns={"model_area_m2": "source_model_area_m2"})


print("Geology polygons prepared:", len(geology_for_area))

print("Model area prepared (km2):", round(geology_for_area["source_model_area_m2"].sum()/ 1_000_000,3))

geology_for_area.head()

Geology polygons prepared: 328
Model area prepared (km2): 79.04


,geology_polygon_id,source_model_area_m2,geometry
0,0,68761.500000,"MULTIPOLYGON Z (((247146 54149 0, 247188 54169..."
1,1,38221.958435,"MULTIPOLYGON Z (((248549 54029 0, 248477 54043..."
2,2,42273.100669,"MULTIPOLYGON Z (((246387 54088 0, 246454 54005..."
3,3,24289.000000,"MULTIPOLYGON Z (((247750 54265 0, 247755 54250..."
4,4,22062.864327,"MULTIPOLYGON Z (((246064 54139 0, 246143 54157..."


## 7. Intersect geothermal geology with LSOA boundaries


In [9]:

lsoa_geology_area = gpd.overlay(geology_for_area,heat_demand[["LSOA_code","LSOA","geometry"]], 
                                how="intersection",keep_geom_type=True)

# Calculate the geothermal model area falling within each LSOA fragment
lsoa_geology_area["fragment_area_m2"] = (lsoa_geology_area.geometry.area)
lsoa_geology_area["fragment_area_km2"] = (lsoa_geology_area["fragment_area_m2"]/ 1_000_000)


## 8. Assign actual uniform grid BHE points directly to LSOAs


In [10]:
#keep the BHE attributes needed for LSOA supply aggregation
bhe_fields_for_lsoa = ["bhe_id","grid_id","grid_spacing_m","model_unit_lithology_id",
                       "low_useful_heat_mwh_year","representative_useful_heat_mwh_year",
                       "high_useful_heat_mwh_year","geometry"]

#spatially assign each BHE point to the LSOA containing it
bhe_lsoa = gpd.sjoin(bhe_grid[bhe_fields_for_lsoa],heat_demand[["LSOA_code","LSOA","geometry"]],
                     how="left",predicate="within")

#remove spatial-join index field
bhe_lsoa = bhe_lsoa.drop(columns=["index_right"]).reset_index(drop=True)

#validate point-to-LSOA assignment
duplicate_assignments = int(bhe_lsoa["bhe_id"].duplicated().sum())
unassigned_bhes = int(bhe_lsoa["LSOA_code"].isna().sum())
assigned_bhes = int(bhe_lsoa["LSOA_code"].notna().sum())

print("Total BHE grid points:",len(bhe_grid))
print("BHEs assigned to an LSOA:",assigned_bhes)
print("BHEs not assigned to an LSOA:",unassigned_bhes)
print("Duplicate BHE assignments:",duplicate_assignments)
print("LSOAs receiving at least one BHE:",bhe_lsoa.loc[bhe_lsoa["LSOA_code"].notna(),"LSOA_code"].nunique())

#check how much representative supply was spatially retained
original_rep_supply_mwh = bhe_grid["representative_useful_heat_mwh_year"].sum()
assigned_rep_supply_mwh = bhe_lsoa.loc[bhe_lsoa["LSOA_code"].notna(),"representative_useful_heat_mwh_year"].sum()

print("Original representative supply (MWh/year):",round(original_rep_supply_mwh,2))
print("Assigned representative supply (MWh/year):",round(assigned_rep_supply_mwh,2))
print("Supply difference (MWh/year):",round(original_rep_supply_mwh-assigned_rep_supply_mwh,2))

# a BHE must never be assigned to more than one LSOA
assert duplicate_assignments == 0,"One or more BHE points were assigned to multiple LSOAs."

bhe_lsoa.head()

Total BHE grid points: 14056
BHEs assigned to an LSOA: 14056
BHEs not assigned to an LSOA: 0
Duplicate BHE assignments: 0
LSOAs receiving at least one BHE: 164
Original representative supply (MWh/year): 398407.4
Assigned representative supply (MWh/year): 398407.4
Supply difference (MWh/year): 0.0


,bhe_id,grid_id,grid_spacing_m,model_unit_lithology_id,low_useful_heat_mwh_year,representative_useful_heat_mwh_year,high_useful_heat_mwh_year,geometry,LSOA_code,LSOA
0,1,1523,75.0,"MDT__SLATE, SILTSTONE AND SANDSTONE",27.234873,29.760611,32.921694,POINT (251362.5 51112.5),E01015130,Plymouth 032C
1,2,1700,75.0,"MDT__SLATE, SILTSTONE AND SANDSTONE",27.234873,29.760611,32.921694,POINT (251362.5 51187.5),E01015130,Plymouth 032C
2,3,1876,75.0,"STG__SANDSTONE, SILTSTONE AND MUDSTONE",25.713659,29.760611,31.797796,POINT (251287.5 51262.5),E01015130,Plymouth 032C
3,4,1877,75.0,"MDT__SLATE, SILTSTONE AND SANDSTONE",27.234873,29.760611,32.921694,POINT (251362.5 51262.5),E01015130,Plymouth 032C
4,5,1878,75.0,"MDT__SLATE, SILTSTONE AND SANDSTONE",27.234873,29.760611,32.921694,POINT (251437.5 51262.5),E01015130,Plymouth 032C


## 9. Aggregate grid-based geothermal supply to LSOA level


In [11]:
#summarise modelled geothermal area by LSOA
lsoa_model_area = (lsoa_geology_area.groupby(["LSOA_code","LSOA"],as_index=False)
                   .agg(geothermal_model_area_m2=("fragment_area_m2","sum")))

#summarise BHE supply by LSOA
lsoa_bhe_supply = (bhe_lsoa.dropna(subset=["LSOA_code"])
                   .groupby(["LSOA_code","LSOA"],as_index=False)
                   .agg(borehole_count=("bhe_id","size"),
                        low_useful_heat_mwh_year=("low_useful_heat_mwh_year","sum"),
                        representative_useful_heat_mwh_year=("representative_useful_heat_mwh_year","sum"),
                        high_useful_heat_mwh_year=("high_useful_heat_mwh_year","sum")))

#combine model area and geothermal supply
lsoa_supply = lsoa_model_area.merge(lsoa_bhe_supply,on=["LSOA_code","LSOA"],how="outer")

supply_fields = ["geothermal_model_area_m2","borehole_count","low_useful_heat_mwh_year",
                 "representative_useful_heat_mwh_year","high_useful_heat_mwh_year"]

lsoa_supply[supply_fields] = lsoa_supply[supply_fields].fillna(0)
lsoa_supply["borehole_count"] = lsoa_supply["borehole_count"].astype(int)
lsoa_supply["geothermal_model_area_km2"] = lsoa_supply["geothermal_model_area_m2"] / 1_000_000

#check allocated supply totals
print("LSOAs with geothermal supply:", len(lsoa_supply))
print("Total model area allocated (km²):", round(lsoa_supply["geothermal_model_area_km2"].sum(),3))
print("Total BHEs allocated:", lsoa_supply["borehole_count"].sum())
print("Representative useful heat allocated (MWh/year):", round(lsoa_supply["representative_useful_heat_mwh_year"].sum(),2))

assert lsoa_supply["borehole_count"].sum() == len(bhe_grid)
assert np.isclose(lsoa_supply["representative_useful_heat_mwh_year"].sum(),bhe_grid["representative_useful_heat_mwh_year"].sum())

lsoa_supply.head()


LSOAs with geothermal supply: 164
Total model area allocated (km²): 79.04
Total BHEs allocated: 14056
Representative useful heat allocated (MWh/year): 398407.4


,LSOA_code,LSOA,geothermal_model_area_m2,borehole_count,low_useful_heat_mwh_year,representative_useful_heat_mwh_year,high_useful_heat_mwh_year,geothermal_model_area_km2
0,E01015023,Plymouth 003A,918876.594890,164,4089.013169,4587.096273,4930.021107,0.918877
1,E01015024,Plymouth 004A,395108.755498,71,1793.301792,1992.729096,2106.751684,0.395109
2,E01015025,Plymouth 001A,757984.507350,136,3404.589962,3935.512826,4149.651166,0.757985
3,E01015026,Plymouth 003B,240628.901274,43,1071.555506,1176.250251,1248.653886,0.240629
4,E01015027,Plymouth 002A,673763.646930,117,2926.045071,3362.244324,3550.394812,0.673764


## 10. Join geothermal supply to LSOA heat demand


In [12]:
#join geothermal supply to heat demand
lsoa_comparison = heat_demand[["LSOA_code","LSOA","total_useful_heat_mwh","geometry"]].merge(
    lsoa_supply.drop(columns="LSOA"),on="LSOA_code",how="left")

#fill LSOAs with no mapped geothermal supply
supply_columns = ["geothermal_model_area_m2","geothermal_model_area_km2","borehole_count",
                  "low_useful_heat_mwh_year","representative_useful_heat_mwh_year","high_useful_heat_mwh_year"]

lsoa_comparison[supply_columns] = lsoa_comparison[supply_columns].fillna(0)

print("Total LSOAs:", len(lsoa_comparison))
print("LSOAs with zero geothermal supply:", int((lsoa_comparison["representative_useful_heat_mwh_year"] == 0).sum()))
print("Representative useful heat retained (MWh/year):", round(lsoa_comparison["representative_useful_heat_mwh_year"].sum(),0))


Total LSOAs: 164
LSOAs with zero geothermal supply: 0
Representative useful heat retained (MWh/year): 398407.0


## 11. Calculate LSOA supply-demand balance


In [13]:
demand_col = "total_useful_heat_mwh"

for thermal_scenario in ["low","representative","high"]:
    supply_col = f"{thermal_scenario}_useful_heat_mwh_year"
    matched_col = f"{thermal_scenario}_matched_heat_mwh_year"
    ratio_col = f"{thermal_scenario}_supply_demand_ratio_pct"
    contribution_col = f"{thermal_scenario}_local_contribution_pct"
    residual_col = f"{thermal_scenario}_residual_demand_mwh_year"
    surplus_col = f"{thermal_scenario}_surplus_mwh_year"

    lsoa_comparison[ratio_col] = np.where(lsoa_comparison[demand_col] > 0,lsoa_comparison[supply_col] / lsoa_comparison[demand_col] * 100,np.nan)
    lsoa_comparison[matched_col] = np.minimum(lsoa_comparison[supply_col],lsoa_comparison[demand_col])
    lsoa_comparison[contribution_col] = np.where(lsoa_comparison[demand_col] > 0,lsoa_comparison[matched_col] / lsoa_comparison[demand_col] * 100,np.nan)
    lsoa_comparison[residual_col] = np.maximum(lsoa_comparison[demand_col] - lsoa_comparison[supply_col],0)
    lsoa_comparison[surplus_col] = np.maximum(lsoa_comparison[supply_col] - lsoa_comparison[demand_col],0)

lsoa_comparison[["LSOA_code","LSOA","total_useful_heat_mwh","representative_useful_heat_mwh_year",
                 "representative_supply_demand_ratio_pct","representative_local_contribution_pct",
                 "representative_residual_demand_mwh_year","representative_surplus_mwh_year"]].head(20)


,LSOA_code,LSOA,total_useful_heat_mwh,representative_useful_heat_mwh_year,representative_supply_demand_ratio_pct,representative_local_contribution_pct,representative_residual_demand_mwh_year,representative_surplus_mwh_year
0,E01034161,Plymouth 034D,5099.178833,1024.623970,20.093901,20.093901,4074.554863,0.0
1,E01034162,Plymouth 034E,5055.913662,1843.334307,36.458975,36.458975,3212.579355,0.0
2,E01034160,Plymouth 034C,5847.675300,1968.381358,33.660921,33.660921,3879.293942,0.0
3,E01034159,Plymouth 034B,4016.406506,1398.305529,34.814841,34.814841,2618.100977,0.0
4,E01015155,Plymouth 034A,4592.201681,1058.159915,23.042540,23.042540,3534.041766,0.0
5,E01015153,Plymouth 033C,6023.391662,1153.066903,19.143150,19.143150,4870.324758,0.0
6,E01015152,Plymouth 027C,9165.928209,1589.388785,17.340184,17.340184,7576.539423,0.0
7,E01015172,Plymouth 033B,6020.165450,1896.788517,31.507249,31.507249,4123.376933,0.0
8,E01015151,Plymouth 033A,6046.113656,905.400539,14.974918,14.974918,5140.713116,0.0
9,E01015051,Plymouth 023E,5436.035344,963.146970,17.717820,17.717820,4472.888374,0.0


## 12. Summarise Plymouth-wide supply-demand balance


In [14]:
total_demand_mwh = lsoa_comparison["total_useful_heat_mwh"].sum()
city_summary_rows = []

for thermal_scenario in ["low","representative","high"]:
    supply_col = f"{thermal_scenario}_useful_heat_mwh_year"
    matched_col = f"{thermal_scenario}_matched_heat_mwh_year"
    residual_col = f"{thermal_scenario}_residual_demand_mwh_year"
    surplus_col = f"{thermal_scenario}_surplus_mwh_year"

    total_supply_mwh = lsoa_comparison[supply_col].sum()
    total_matched_mwh = lsoa_comparison[matched_col].sum()
    total_residual_mwh = lsoa_comparison[residual_col].sum()
    total_surplus_mwh = lsoa_comparison[surplus_col].sum()

    city_summary_rows.append({"thermal_scenario": thermal_scenario,
                              "total_heat_demand_mwh_year": total_demand_mwh,
                              "total_geothermal_supply_mwh_year": total_supply_mwh,
                              "aggregate_supply_demand_ratio_pct": total_supply_mwh / total_demand_mwh * 100,
                              "spatially_matched_heat_mwh_year": total_matched_mwh,
                              "spatially_matched_contribution_pct": total_matched_mwh / total_demand_mwh * 100,
                              "residual_heat_demand_mwh_year": total_residual_mwh,
                              "surplus_geothermal_potential_mwh_year": total_surplus_mwh})

city_summary = pd.DataFrame(city_summary_rows)
city_summary


,thermal_scenario,total_heat_demand_mwh_year,total_geothermal_supply_mwh_year,aggregate_supply_demand_ratio_pct,spatially_matched_heat_mwh_year,spatially_matched_contribution_pct,residual_heat_demand_mwh_year,surplus_geothermal_potential_mwh_year
0,low,794822.196182,352668.497117,44.370741,322213.410314,40.539055,472608.785867,30455.086803
1,representative,794822.196182,398407.399777,50.125349,355896.873996,44.776917,438925.322185,42510.525781
2,high,794822.196182,424115.070698,53.359742,374525.154259,47.120621,420297.041923,49589.916440


## 13. Identify LSOAs with the highest local geothermal contribution


In [15]:
priority_review = lsoa_comparison[["LSOA_code","LSOA","total_useful_heat_mwh","geothermal_model_area_km2",
                                   "representative_useful_heat_mwh_year","representative_matched_heat_mwh_year",
                                   "representative_supply_demand_ratio_pct","representative_local_contribution_pct",
                                   "representative_residual_demand_mwh_year","representative_surplus_mwh_year"]].copy()

priority_review = priority_review.sort_values("representative_local_contribution_pct",ascending=False)
priority_review.head(10)


,LSOA_code,LSOA,total_useful_heat_mwh,geothermal_model_area_km2,representative_useful_heat_mwh_year,representative_matched_heat_mwh_year,representative_supply_demand_ratio_pct,representative_local_contribution_pct,representative_residual_demand_mwh_year,representative_surplus_mwh_year
82,E01015082,Plymouth 004B,4067.406282,1.067570,5168.056231,4067.406282,127.060241,100.0,0.0,1100.649949
47,E01015145,Plymouth 014E,3566.005134,0.767298,3740.465588,3566.005134,104.892322,100.0,0.0,174.460454
78,E01015023,Plymouth 003A,4451.180235,0.918877,4587.096273,4451.180235,103.053483,100.0,0.0,135.916038
80,E01015083,Plymouth 004C,4105.717109,1.241260,5984.112562,4105.717109,145.750728,100.0,0.0,1878.395453
87,E01015139,Plymouth 031E,4553.879513,1.200029,6242.904666,4553.879513,137.089808,100.0,0.0,1689.025152
113,E01015131,Plymouth 030C,8284.070479,2.625314,13674.614232,8284.070479,165.071196,100.0,0.0,5390.543753
114,E01034157,Plymouth 031G,4666.533278,3.914598,20664.123559,4666.533278,442.815305,100.0,0.0,15997.590281
117,E01015112,Plymouth 019A,4919.005416,1.156418,5858.688557,4919.005416,119.103113,100.0,0.0,939.683141
121,E01015113,Plymouth 019B,4648.931860,1.085168,5579.352709,4648.931860,120.013648,100.0,0.0,930.420850
144,E01015095,Plymouth 010D,3719.927995,1.017411,5184.962658,3719.927995,139.383415,100.0,0.0,1465.034663


## 14. Identify LSOAs with the highest absolute matched geothermal heat


In [16]:
matched_heat_review = lsoa_comparison[["LSOA_code","LSOA","total_useful_heat_mwh","geothermal_model_area_km2",
                                       "representative_useful_heat_mwh_year","representative_matched_heat_mwh_year",
                                       "representative_local_contribution_pct","representative_residual_demand_mwh_year",
                                       "representative_surplus_mwh_year"]].copy()

matched_heat_review = matched_heat_review.sort_values("representative_matched_heat_mwh_year",ascending=False)
matched_heat_review.head(10)


,LSOA_code,LSOA,total_useful_heat_mwh,geothermal_model_area_km2,representative_useful_heat_mwh_year,representative_matched_heat_mwh_year,representative_local_contribution_pct,representative_residual_demand_mwh_year,representative_surplus_mwh_year
158,E01015097,Plymouth 001B,9435.323182,1.783746,9177.461166,9177.461166,97.267057,257.862016,0.000000
113,E01015131,Plymouth 030C,8284.070479,2.625314,13674.614232,8284.070479,100.000000,0.000000,5390.543753
93,E01015182,Plymouth 028E,5675.579941,0.938366,5153.905472,5153.905472,90.808438,521.674469,0.000000
117,E01015112,Plymouth 019A,4919.005416,1.156418,5858.688557,4919.005416,100.000000,0.000000,939.683141
147,E01015091,Plymouth 005B,4847.681677,1.110237,5786.975829,4847.681677,100.000000,0.000000,939.294152
156,E01015092,Plymouth 005C,4822.412835,2.459606,12543.853026,4822.412835,100.000000,0.000000,7721.440192
114,E01034157,Plymouth 031G,4666.533278,3.914598,20664.123559,4666.533278,100.000000,0.000000,15997.590281
145,E01015120,Plymouth 015A,4660.389409,1.738905,8808.461113,4660.389409,100.000000,0.000000,4148.071704
121,E01015113,Plymouth 019B,4648.931860,1.085168,5579.352709,4648.931860,100.000000,0.000000,930.420850
65,E01015029,Plymouth 008A,6190.450448,0.929106,4567.176832,4567.176832,73.777779,1623.273615,0.000000


## 15. Export final supply-demand results


In [17]:
if OUTPUT_GPKG.exists():
    OUTPUT_GPKG.unlink()

#save final LSOA supply-demand outputs
lsoa_comparison.to_file(OUTPUT_GPKG,layer="lsoa_supply_demand_2024",driver="GPKG")
lsoa_comparison.drop(columns="geometry").to_csv(LSOA_COMPARISON_CSV,index=False)
city_summary.to_csv(CITY_SUMMARY_CSV,index=False)

In [ ]:
#export LSOA supply-demand table
TABLE_OUTPUT_DIR = PROJECT_DIR / "03_Outputs/Tables"
TABLE_OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

lsoa_results_table = (lsoa_comparison.copy()
                      .sort_values(["representative_local_contribution_pct","representative_matched_heat_mwh_year"],ascending=False))
lsoa_results_table["LSOA code and name"] = lsoa_results_table["LSOA_code"] + " " + lsoa_results_table["LSOA"]
lsoa_results_table["Residential useful heat demand (GWh yr-1)"] = (lsoa_results_table["total_useful_heat_mwh"] / 1_000).round(2)
lsoa_results_table["Heat-demand density (GWh km-2 yr-1)"] = ((lsoa_results_table["total_useful_heat_mwh"] / 1_000) / (lsoa_results_table.geometry.area / 1_000_000)).round(2)
lsoa_results_table["Candidate BHE count"] = lsoa_results_table["borehole_count"].round(0).astype(int)
lsoa_results_table["Representative GSHP useful heat potential (GWh yr-1)"] = (lsoa_results_table["representative_useful_heat_mwh_year"] / 1_000).round(2)
lsoa_results_table["Locally matched GSHP useful heat (GWh yr-1)"] = (lsoa_results_table["representative_matched_heat_mwh_year"] / 1_000).round(2)
lsoa_results_table["Local heat-demand coverage (%)"] = lsoa_results_table["representative_local_contribution_pct"].round(1)
lsoa_results_table["Unmet residential heat demand (GWh yr-1)"] = (lsoa_results_table["representative_residual_demand_mwh_year"] / 1_000).round(2)
lsoa_results_table["Surplus GSHP potential (GWh yr-1)"] = (lsoa_results_table["representative_surplus_mwh_year"] / 1_000).round(2)

lsoa_results_table = lsoa_results_table[["LSOA code and name","Residential useful heat demand (GWh yr-1)","Heat-demand density (GWh km-2 yr-1)",
                               "Candidate BHE count","Representative GSHP useful heat potential (GWh yr-1)","Locally matched GSHP useful heat (GWh yr-1)",
                               "Local heat-demand coverage (%)","Unmet residential heat demand (GWh yr-1)","Surplus GSHP potential (GWh yr-1)"]]
lsoa_results_table.to_csv(TABLE_OUTPUT_DIR / "LSOA_Supply_Demand_Results.csv",index=False)